# reglscatterpy — full feature tour

The Python companion to **reglScatterplotR**. By default it renders the *same* compiled widget the R package ships (legend, lasso, tooltips, sync, PNG/SVG/PDF export) via [anywidget](https://anywidget.dev), so plots are pixel-identical across R and Python and work in **Jupyter, JupyterLab, VS Code and Colab**.

```bash
pip install -e .            # from the python/ dir; pulls numpy, pandas, anywidget
pip install anndata         # for the AnnData examples
```

Run cells top-to-bottom. Each `scatterplot(...)` call displays an interactive widget.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import reglscatterpy as rs
print('reglscatterpy', rs.__version__)

A small synthetic single-cell-style DataFrame (mirrors the R `reglScatterExample`).

In [ ]:
rng = np.random.default_rng(0)
n = 3000
ct = rng.choice(['CD4 T','CD8 T','NK','B','Monocyte','Dendritic'], size=n)
centers = {k:(rng.uniform(-6,6), rng.uniform(-6,6)) for k in set(ct)}
umap1 = np.array([centers[c][0] for c in ct]) + rng.normal(0,1,n)
umap2 = np.array([centers[c][1] for c in ct]) + rng.normal(0,1,n)
df = pd.DataFrame({'UMAP_1':umap1, 'UMAP_2':umap2, 'celltype':ct,
                   'nCount':rng.integers(2000,12000,n), 'CD3D':rng.gamma(2,1,n)})
df.head()

## 1. Basics

### 1.1 Minimal plot (DataFrame)

In [ ]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2')

### 1.2 Colour by a categorical column

In [ ]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='celltype', legend_title='Cell type')

### 1.3 Colour by a continuous column

In [ ]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='CD3D', continuous_palette='viridis', legend_title='CD3D')

### 1.4 Fixed point colour

In [ ]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', point_color='#0072B2')

## 2. Point appearance

### 2.1 Size & opacity

In [ ]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='celltype', point_size=6, opacity=0.5)

### 2.2 `pixel_ratio` (crisp / export-quality)

In [ ]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='celltype', pixel_ratio=3)

## 3. Palettes

Continuous (viridis family) and categorical (ColorBrewer) palettes are byte-identical to the R package.

In [ ]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='nCount', continuous_palette='magma', legend_title='nCount')

### 3.1 Custom per-category colours

In [ ]:
my = {c:h for c,h in zip(sorted(set(ct)), ['#e6194B','#3cb44b','#4363d8','#f58231','#911eb4','#42d4f4'])}
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='celltype', custom_colors=my)

## 4. Continuous scaling: `vmin` / `vmax` / `center_zero`

In [ ]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='CD3D', vmin=0, vmax='p95')

## 5. Filtering

`filter_by` adds interactive numeric range filters.

In [ ]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='celltype', filter_by=df[['nCount','CD3D']])

## 6. Titles, axes, theming & legend

In [ ]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='celltype',
    title='PBMC — UMAP', xlab='UMAP 1', ylab='UMAP 2',
    background_color='#111418', axis_color='#cccccc',
    legend_bg='#1c2128', legend_text='#e6edf3', legend_position='bottom-left')

## 7. Export (PNG / SVG / PDF)

In [ ]:
rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='celltype', enable_download=True)

## 8. AnnData / scanpy

Coordinates from `obsm`, colour from an `obs` column or a gene in `var_names`.

In [ ]:
try:
    import anndata as ad
    g = 30
    X = rng.poisson(2, size=(n, g)).astype(float)
    adata = ad.AnnData(X=X,
        obs=pd.DataFrame({'celltype': pd.Categorical(ct)}),
        var=pd.DataFrame(index=[f'Gene{i}' for i in range(g)]))
    adata.obsm['X_umap'] = np.c_[umap1, umap2]
    display(rs.scatterplot(adata, x='umap', color_by='celltype'))   # obsm + obs column
except ModuleNotFoundError:
    print('anndata not installed — pip install anndata')

### 8.1 Colour by a gene (read from `.X`)

In [ ]:
try:
    import anndata  # noqa
    display(rs.scatterplot(adata, x='umap', color_by='Gene0', continuous_palette='viridis'))
except (ModuleNotFoundError, NameError):
    print('anndata not available')

## 9. numpy array input

In [ ]:
emb = np.c_[umap1, umap2]
rs.scatterplot(emb, color_by=ct)        # first two columns are x/y

## 10. MuData / SpatialData (optional)

In [ ]:
# rs.scatterplot(mdata, x='rna:X_umap', color_by='rna:celltype')   # MuData
# rs.scatterplot(sdata, table='table', x='spatial', color_by='region')  # SpatialData
print('See docstring: rs.scatterplot.__doc__')

## 11. Alternative backend: jupyter-scatter

`backend='jscatter'` renders through jupyter-scatter (no package legend/sync/export UI). Needs `pip install reglscatterpy[render]`.

In [ ]:
# rs.scatterplot(df, x='UMAP_1', y='UMAP_2', color_by='celltype', backend='jscatter')

## 12. Inspecting the extracted data

In [ ]:
from reglscatterpy import extract
pd_data = extract(df, x='UMAP_1', y='UMAP_2', color_by='celltype')
print('n =', pd_data.n, '| color_name =', pd_data.color_name)
pd_data.x[:5], pd_data.y[:5]